# 7. Evaluation with DeepEval

Score a RAG pipeline with DeepEval's LLM-as-judge metrics, using a local Ollama model as the judge.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

You cannot improve what you cannot measure, and eyeballing five answers is not measurement. **DeepEval** is a pytest-style library that scores RAG outputs with LLM-as-judge metrics. Learning it early means every later technique in this course can be checked with numbers. We plug in a local Ollama model as the judge so no API key is needed.

## How it works

1. Wrap the local model in `DeepEvalBaseLLM`
2. Create `LLMTestCase`s: input, actual output, expected output, retrieval context
3. Pick metrics: answer relevancy, faithfulness, contextual relevancy
4. Call `metric.measure(case)` and read `score`, `reason`
5. Aggregate across the golden set

```text
 question --> RAG --> (answer, retrieved context)
                           |
          LLMTestCase(input, actual_output, expected_output, retrieval_context)
                           |
   AnswerRelevancy   Faithfulness   ContextualRelevancy   --> scores + reasons
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install deepeval`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase

class LocalJudge(DeepEvalBaseLLM):
    """DeepEval judge backed by Ollama. When DeepEval passes a pydantic `schema`, we use Ollama's
    structured-output mode so a small local model is forced to return valid JSON."""

    def load_model(self):
        return LLM_MODEL

    def generate(self, prompt: str, schema=None):
        import ollama
        if schema is None:
            return llm(prompt)
        reply = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}],
                            format=schema.model_json_schema(), options={"temperature": 0})
        return schema.model_validate_json(reply["message"]["content"])

    async def a_generate(self, prompt: str, schema=None):
        return self.generate(prompt, schema)

    def get_model_name(self):
        return LLM_MODEL

judge = LocalJudge()
metrics = [
    AnswerRelevancyMetric(model=judge, threshold=0.7),
    FaithfulnessMetric(model=judge, threshold=0.7),
    ContextualRelevancyMetric(model=judge, threshold=0.5),
]

Build one test case per golden question: the question, our answer, and the retrieved context.

In [ ]:
cases = []
for q in load_eval_questions()[:4]:
    response, hits = rag(q["question"], store, k=3)
    cases.append(LLMTestCase(input=q["question"], actual_output=response,
                             expected_output=q["answer"], retrieval_context=[h["text"] for h in hits]))

In [ ]:
for case in cases:
    print("Q:", case.input)
    for metric in metrics:
        try:
            metric.measure(case)
            print(f"  {type(metric).__name__:28s} {metric.score:.2f}  {'PASS' if metric.is_successful() else 'FAIL'}")
        except Exception as e:   # small local judges occasionally break DeepEval's parsing
            print(f"  {type(metric).__name__:28s} judge error: {str(e)[:80]}")

Small local judges are noisy - run each metric several times or use a stronger judge for decisions that matter.

## Read the judge's reasoning
Every metric returns a `reason`. Reading reasons is how you debug a low score.

In [ ]:
case = cases[0]
for m in metrics:
    try:
        m.measure(case)
    except Exception as e:
        print(f"{type(m).__name__}: judge error {str(e)[:80]}"); continue
    print(f"{type(m).__name__}: {m.score:.2f}\n  reason: {getattr(m, 'reason', None)}\n")

## Evaluate a deliberately bad pipeline
A useful metric must separate good from bad. Break the pipeline (retrieve irrelevant chunks) and confirm the scores drop.

In [ ]:
bad_ctx = [chunks[-1], chunks[-2]]                       # irrelevant chunks
bad_case = LLMTestCase(input=cases[0].input, actual_output=answer(cases[0].input, bad_ctx),
                       expected_output=cases[0].expected_output, retrieval_context=bad_ctx)
for m in metrics:
    try:
        m.measure(bad_case)
        print(f"{type(m).__name__:28s} bad pipeline: {m.score:.2f}")
    except Exception as e:
        print(f"{type(m).__name__:28s} judge error: {str(e)[:80]}")

## Aggregate over the golden set

In [ ]:
import pandas as pd
rows = []
for case in cases:
    row = {"question": case.input[:50]}
    for m in metrics:
        try:
            m.measure(case); row[type(m).__name__.replace("Metric", "")] = round(m.score, 2)
        except Exception as e:
            row[type(m).__name__.replace("Metric", "")] = None
    rows.append(row)
report = pd.DataFrame(rows).set_index("question")
display(report); print(report.mean().round(2))

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Standard metrics with explanations | Small local judges are noisy and slow |
| Works with any model as judge | Metrics are LLM opinions, not ground truth |
| Integrates with pytest / CI | Needs structured outputs that tiny models sometimes fail to produce |

## When to use it

From day one. Use a strong judge model for decisions and a local one for quick iteration.

## Experiments to try

- Add `ContextualRecallMetric` and `ContextualPrecisionMetric`.
- Run each metric three times; how much do scores vary?
- Wire the metrics into a pytest test that fails if faithfulness < 0.7.

## Key takeaways

- Score retrieval and generation separately.
- Read the `reason` field.
- Validate the judge by feeding it bad pipelines.

---
**Next:** [8. Query Transformations](../08_query_transformations/query_transformations.ipynb)